In [95]:
import pandas as pd
import pymysql
import unicodedata
import re
from nltk.corpus import stopwords
import string
from collections import Counter

In [96]:
base_registro_civil = pd.read_parquet(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\bases\base_rucs_sri.parquet")

In [97]:
def quitar_tildes(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

def limpiador(lista: list):
    documentos = []
    re_punctuation = re.compile('[%s]' % re.escape(string.punctuation))

    stop_words_s = set(stopwords.words('spanish'))
    stop_words = set(stopwords.words('english'))

    for descripcion in lista:
        tokens = descripcion.split()
        tokens = [re_punctuation.sub(' ', w) for w in tokens]
        tokens = ' '.join(tokens).split()

        tokens = [quitar_tildes(word.lower()) for word in tokens]

        tokens = [word.lower() for word in tokens if re.search('[a-z ]', word.lower())]
        tokens = [word.lower() for word in tokens if word.isalpha()]
        
        tokens = [w for w in tokens if w not in stop_words_s]
        tokens = [w for w in tokens if w not in stop_words]
        
        tokens = [word for word in tokens if len(word) > 2]

        if len(tokens) > 0:
            documento = ' '.join(tokens)
        else:
            documento = ''  # ← asegura longitud igual al DF

        documentos.append(documento)

    return documentos

In [98]:
#Nos quedamos solo con los rucs interesantes

base_registro_civil['nombre_fantasia_comercial'] = (
    base_registro_civil['nombre_fantasia_comercial']
        .replace("", pd.NA)
)

base_registro_civil = base_registro_civil[base_registro_civil["nombre_fantasia_comercial"].notna()]
base_registro_civil['motivo_cancelacion_suspension'] = (base_registro_civil['motivo_cancelacion_suspension'].replace("", pd.NA))
base_registro_civil = base_registro_civil[base_registro_civil['motivo_cancelacion_suspension'].isna()]

In [99]:
recreo_nombres = pd.read_excel(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\tablas_locales_CC\ccrecreo.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: "C:\\Users\\anali\\OneDrive - PUBLIPROMUEVE S.A\\Ruben Freire's files - CENTROS COMERCIALES\\sandbox\\tablas_locales_CC\\ccrecreo.xlsx"

In [ ]:
recreo_nombres= pd.read_excel(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\tablas_locales_CC\Tabla maestro Condado 3 3.xlsx")

In [ ]:
recreo_nombres.columns

Index(['num', 'nombre_establecimiento', 'RUC', 'CÓDIGO ESTABLECIMIENTO',
       'ESTADO', 'CÓDIGO ESTABLECIMIENTO.1', 'id_establecimiento', 'categoria',
       'tipo_local', 'posible'],
      dtype='object')

In [ ]:
#Normalizamos los q gramas
def normalizar(s):
    if pd.isna(s):
        return ""
    s = s.lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

In [ ]:
#parte una palabra en la cantidad de q gramas dados
def qgrams(s, q=3):
    return {s[i:i+q] for i in range(len(s) - q + 1)}

In [ ]:
#Es la medida que vamos a tomar. La cantidad de q gramas dados los que coinciden dividido para todos
def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


In [ ]:
dicc_nombre_fantasia = set(recreo_nombres['nombre_establecimiento'])

In [ ]:
# Normalizamos los nombres fantasia
dic_norm = {normalizar(x): x for x in dicc_nombre_fantasia}

dic_qgrams = {
    k: qgrams(k, q=3)
    for k in dic_norm.keys()
}

In [ ]:
#Match de los qgramas en los centros comerciales
# def match_qgram(nombre,centro_comercial, threshold=0.87, umbral_centro_comercial = 0.28):
#     nombre_sin_centro_comercial = nombre.lower().replace(centro_comercial, "").strip()
#     s = normalizar(nombre_sin_centro_comercial)
#     q_s = qgrams(s)

#     mejor, score = None, 0
#     for k, q_k in dic_qgrams.items():
#         sim = jaccard(q_s, q_k)
#         if sim > score:
#             mejor, score = dic_norm[k], sim

#     if score >= threshold:
#         return mejor, score
#     elif (score>= umbral_centro_comercial) and (centro_comercial.lower() in nombre.lower()):
#         return mejor , 1 
#     return None, score

In [ ]:
def levenshtein_similarity(s1, s2):
    # Calcula un score de 0 a 1 basado en la distancia
    max_len = max(len(s1), len(s2))
    if max_len == 0: return 1.0
    return 1 - (lev(s1, s2) / max_len)

def match_qgram(nombre:str,
                centro_comercial: str, 
                threshold: float, 
                threshold_cc: float, 
                umbral_superior_segundo_filtro: float, 
                umbral_inferior_segundo_filtro:float)-> [str, float]:
    
    nombre_sin_cc = normalizar(nombre.lower().replace(centro_comercial, "").strip())
    q_s = qgrams(nombre_sin_cc)

    mejor, mejor_score_q = None, 0
    
    # 1. Fase de Q-Grams (Filtro rápido)
    for k, q_k in dic_qgrams.items():
        sim = jaccard(q_s, q_k)
        if sim > mejor_score_q:
            mejor, mejor_score_q = k, sim # Guardamos la clave para procesar luego

    # 2. Lógica de Decisión
    
    # Match casi perfecto: No perdemos tiempo en más cálculos
    if mejor_score_q >= umbral_superior_segundo_filtro:
        return dic_norm[mejor], mejor_score_q

    # Zona Gris: Aquí es donde Levenshtein brilla (ej. Tropiburger vs Tropiburguer)
    if umbral_inferior_segundo_filtro <= mejor_score_q < umbral_superior_segundo_filtro:
        score_lev = levenshtein_similarity(nombre_sin_cc, mejor)
        
        if score_lev >= threshold:
            return mejor, score_lev

    # 3. Caso especial de Centro Comercial (tu regla de negocio)
    if (mejor_score_q >= threshold_cc) and (centro_comercial.lower() in nombre.lower()):
        return mejor, 1.0

    return None, mejor_score_q

In [ ]:
#Normalizamos el nombre fantasía comercial
base_registro_civil['nombre_fantasia_comercial'] = base_registro_civil['nombre_fantasia_comercial'].apply(normalizar)

#Tenemos la direccion completa separada en base registro civil
base_registro_civil[['provincia',  'canton', 'parroquia', 'calles']] = (
    base_registro_civil['direccion_completa']
        .str.split('/', n=3, expand=True)
)

In [ ]:
#Para sacarnos las frecuencias

#Nombre del centro comercial
nombre = 'csacasc'
apellido = 'condado'
rejex = f'(?=.*{apellido})'

#Filtros para las calles
mask_regex = base_registro_civil['nombre_fantasia_comercial'].str.contains(rf'{rejex}', case = False, na = False)
mask_canton = base_registro_civil['canton'].str.contains("Quito", case = False, na =  False)
mask_parroquia =  base_registro_civil['parroquia'].str.contains("ponce", case = False, na =  False)

#Filtramos la base para obtener las calles
base_filtrada = base_registro_civil[mask_regex]

#Obtenemos las calles
list_calles =list(base_filtrada['calles'])

#Limpiamos las calles y hacemos una sola list
list_calles_limpia = limpiador(list_calles)
list_calles_limpia_total = [
    palabra
    for i in range(len(list_calles_limpia))
    for palabra in list_calles_limpia[i].split()
]

#Sacamos las frecuencias
frecuencias = Counter(list_calles_limpia_total)

#Hacemos dataframe de frecuencias
df_frecuencias = ( 
    pd.DataFrame(frecuencias.items(), columns = ['palabra', 'frecuencia'])
    .sort_values('frecuencia', ascending = False)
    .reset_index(drop = True)
)


In [ ]:

#Guardamos las frecuencias
#df_frecuencias.to_excel(rf"frecuencias_{nombre}_{apellido}.xlsx", index = False)

In [ ]:
lista_palabras_acotacion = ["antonio", "jose", "sucre", "prensa", "john", "kennedy", "Leonardo", "davinci", "mariscal", "sucre"]
lista_precisa  = ['av', "san", "cardenas", "caton","procel", "juan"]

lista_acotacion_especifica = lista_palabras_acotacion +lista_precisa

str_clave = "|".join(lista_palabras_acotacion)

In [ ]:
mask_canton = base_registro_civil['canton'].str.contains("QUITO", case = False, na = False) 
mask_parroquia = base_registro_civil['parroquia'].str.contains("ponceano|cotocollao", case = False, na = False)
#mask_calle_mald  = base_registro_civil['calles'].str.contains("", case = False, na = False)
mask_calles = base_registro_civil['calles'].str.contains(f"{str_clave}", case = False, na = False)

mask_general = mask_canton & (mask_parroquia & mask_calles)

In [ ]:
base_registro_civil_test = base_registro_civil[mask_general].copy()

In [ ]:
base_registro_civil_test[['matcheo_qgram', 'score_qgram']] = (
    base_registro_civil_test['nombre_fantasia_comercial']
      .apply(lambda x: pd.Series(match_qgram(x, centro_comercial='condado', threshold=0.9, threshold_cc=0.4, umbral_superior_segundo_filtro=0.9, umbral_inferior_segundo_filtro=0.4)))
)

In [ ]:

def porcentaje_coincidencias(calle: str):
    num_coincidencia = 0
    
    # Unificamos a minúsculas para comparar correctamente
    join_acotacion_general = " ".join(lista_palabras_acotacion).lower()
    join_cotacion_especifica = " ".join(lista_precisa).lower()
    
    # Limpiamos y dividimos el string de la calle
    split_calle = [p for p in re.split(r"[.\s]+", calle.lower()) if p]
    
    if not split_calle:
        return 0.0

    for palabra in split_calle:
        if palabra in join_cotacion_especifica:
            num_coincidencia += 2
        elif palabra in join_acotacion_general:
            num_coincidencia += 1
                
    return round(num_coincidencia, 2) # Redondeado a 2 decimales

In [ ]:
# Cantidad porcentaje match que hay
total_palabras = len(lista_acotacion_especifica)
base_registro_civil_test['cantidad_match_calle'] = (base_registro_civil_test['calles'].apply(porcentaje_coincidencias))

In [ ]:
# Normalizacion [0,2] y score general
base_registro_civil_test["score_qgram_02"] =2*base_registro_civil_test['score_qgram']

base_registro_civil_test["porcentaje_match_02"] =2*base_registro_civil_test['cantidad_match_calle']/base_registro_civil_test['cantidad_match_calle'].max()
#Score General
base_registro_civil_test['score_general'] = base_registro_civil_test["porcentaje_match_02"]*base_registro_civil_test["score_qgram_02"]

In [ ]:
base_registro_civil_test = base_registro_civil_test.sort_values(by = "score_general", ascending = False) 

In [118]:
base_registro_civil_test[base_registro_civil_test['nombre_fantasia_comercial'].str.contains("joan")]

,id_establecimiento,numero_ruc,numero_establecimiento,razon_social,nombre_fantasia_comercial,cod_estado_contribuyente,estado_contribuyente,cod_estado_establecimiento,estado_establecimiento,matriz,...,provincia,canton,parroquia,calles,matcheo_qgram,score_qgram,cantidad_match_calle,score_qgram_02,porcentaje_match_02,score_general
5564392,1.704912e+15,1.704912e+12,5.0,CABRERA PAREDES CARLOS ALEJANDRO,joan stevens,1.0,ACTIVO,2.0,CERRADO,0,...,PICHINCHA,QUITO,COTOCOLLAO,AV. LA PRENSA S/N Y AV. MARISCAL SUCRE,JOAN STEVENS,1.000000,8,2.000000,1.142857,2.285714
6605812,1.720639e+15,1.720639e+12,1.0,CELA SARMIENTO JOAN SEBASTIAN,mega despensa joan s,1.0,ACTIVO,1.0,ABIERTO,1,...,PICHINCHA,QUITO,COTOCOLLAO,JOSE H FIGUEROA OE8-226 Y PASAJE K,NaN,0.217391,4,0.434783,0.571429,0.248447


In [122]:
filtrada_score = base_registro_civil_test[(base_registro_civil_test['matcheo_qgram'].notna())][['numero_establecimiento','numero_ruc','matcheo_qgram', 'score_qgram_02', 'nombre_fantasia_comercial','calles', 'porcentaje_match_02', 'score_general']]

In [ ]:
filtrada_score[filtrada_score['matcheo_qgram'].str.contains("jo", case = False, na = False)]

,numero_establecimiento,numero_ruc,matcheo_qgram,score_qgram_02,nombre_fantasia_comercial,calles,porcentaje_match_02,score_general
5564392,5.0,1.704912e+12,JOAN STEVENS,2.0,joan stevens,AV. LA PRENSA S/N Y AV. MARISCAL SUCRE,1.142857,2.285714
7255193,3.0,1.792190e+12,CORPORACION EUGENIO ESPEJO,2.0,corporacion eugenio espejo,AV. LAPRENSA S/N Y AV. MARISCAL SUCRE,1.000000,2.000000
7341857,6.0,1.793083e+12,MR JOY,2.0,mr joy,AV. MARISCAL SUCRE Y JHON F KENNEDY,0.857143,1.714286
7275916,7.0,1.792379e+12,MR JOY,2.0,mr joy,AV MARISCAL SUCRE SN Y JHON F KENNEDY,0.857143,1.714286


: 

In [115]:
recreo_nombres[recreo_nombres['nombre_establecimiento'].str.contains("jo", case = False, na = False)]

,num,nombre_establecimiento,RUC,CÓDIGO ESTABLECIMIENTO,ESTADO,CÓDIGO ESTABLECIMIENTO.1,id_establecimiento,categoria,tipo_local,posible
51,52,CORPORACION EUGENIO ESPEJO,1792189616001,3,ABIERTO,003,1792189616001003,HOGAR,LOCAL REGULAR,NaN
87,88,JOAN STEVENS,1704911500001,6,ABIERTO,006,1704911500001006,MODA,LOCAL REGULAR,NaN
88,89,JOMATIK (fossil),102311594001,18,CERRADO,018,102311594001018,JOYERÍA,LOCAL REGULAR,NaN
89,90,JOYERIA CASTRO,1711193951001,4,ABIERTO,004,1711193951001004,JOYERÍA,LOCAL REGULAR,NaN
129,130,MR JOY,1792379253001,7,ABIERTO,007,1792379253001007,ENTRETENIMIENTO,LOCAL REGULAR,SI


In [ ]:
base_registro_civil[base_registro_civil['nombre_fantasia_comercial'].str.contains("pandora", case = False, na = False)].sort_values(by = 'numero_establecimiento')

,id_establecimiento,numero_ruc,numero_establecimiento,razon_social,nombre_fantasia_comercial,cod_estado_contribuyente,estado_contribuyente,cod_estado_establecimiento,estado_establecimiento,matriz,...,fecha_actualizacion_comercio,nombre_representante_legal,identificacion_representante_legal,representantes_legales,fecha_actualizacion,encontrado,provincia,canton,parroquia,calles
29810,1.010241e+14,1.010241e+11,1.0,DURAN DURAN RUTH MARCELA DE LOS DOLORES,pandora shop,1.0,ACTIVO,1.0,ABIERTO,1,...,2023-03-20,,,[],2026-01-30 05:55:40,1,AZUAY,CUENCA,HUAYNACAPAC,ALFONSO CORDERO 2-65 Y JOSE PERALTA
57191,1.016252e+14,1.016252e+11,1.0,ABAD CHAVEZ CARMEN BEATRIZ,pandora spa,1.0,ACTIVO,1.0,ABIERTO,1,...,2024-01-08,,,[],2025-11-29 05:25:40,1,AZUAY,CUENCA,EL SAGRARIO,MARISCAL LAMAR 7-23 Y ANTONIO BORRERO
202783,1.045363e+14,1.045363e+11,1.0,MACHUCA CONTRERAS ROQUE STALIN,pandora s box 3ditora digital,1.0,ACTIVO,1.0,ABIERTO,1,...,2019-03-26,,,[],2025-12-07 23:59:19,1,AZUAY,CUENCA,HERMANO MIGUEL,DANIEL CAÑIZARES S/N Y BARON DE COUBERTAIN
265480,1.059641e+14,1.059641e+11,1.0,PADILLA URDIALES PAOLA BERNABE,secretos de pandora,1.0,ACTIVO,1.0,ABIERTO,1,...,None,,,[],2026-01-25 13:10:16,1,AZUAY,GUALACEO,REMIGIO CRESPO TORAL (GULAG),S/N
354471,1.951166e+14,1.951166e+11,1.0,FUNDACION LAS HIJAS DE PANDORA,las hijas de pandora,1.0,ACTIVO,1.0,ABIERTO,1,...,None,JAUREGUI TAMA CONSTANZA,0106418700,"[{""nombre"": ""JAUREGUI TAMA CONSTANZA"", ""identi...",2025-11-21 08:29:40,1,AZUAY,CUENCA,NULTI,VIA A BURIÑA S/N
827683,5.026213e+14,5.026213e+11,1.0,NAVARRO FABARA JESSICA LISSETH,pandora lab,1.0,ACTIVO,1.0,ABIERTO,1,...,2025-05-28,,,[],2026-01-24 21:00:10,1,PICHINCHA,QUITO,MARISCAL SUCRE,E4E RODRIGO DE TRIAN N26-21 Y AV FCO DE ORELLANA
883485,5.036992e+14,5.036992e+11,1.0,CAJAMARCA PALOMO NELLY GABRIELA,calzado la caja de pandora,1.0,ACTIVO,1.0,ABIERTO,1,...,None,,,[],2026-01-19 02:07:51,1,COTOPAXI,SALCEDO,SAN MIGUEL,ROCAFUERTE Y GONZALEZ SUAREZ
1804046,9.008420e+14,9.008420e+11,1.0,GORDON SOLORZANO LIDIA BOLIVIA,pandora coffee house,1.0,ACTIVO,1.0,ABIERTO,1,...,2022-07-28,,,[],2025-11-29 11:35:31,1,GUAYAS,GUAYAQUIL,TARQUI,DATILES S/N Y LA PRIMERA
3515938,9.928725e+14,9.928725e+11,1.0,PANDORA SOLUTIONS S.A. PANDORASA,pandorasa,1.0,ACTIVO,1.0,ABIERTO,1,...,2023-12-19,KOEHN SANTISTEVAN DIETER GERARDO,0916029200,"[{""nombre"": ""KOEHN SANTISTEVAN DIETER GERARDO""...",2026-01-24 07:00:33,1,GUAYAS,SAMBORONDON,LA PUNTILLA (SATELITE),SN 29-30 Y SN
2839439,9.253539e+14,9.253539e+11,1.0,BAGUA LEON DOLORES JUDITH,vivero pandora,1.0,ACTIVO,1.0,ABIERTO,1,...,2023-03-27,,,[],2026-01-28 04:53:32,1,GUAYAS,DURAN,ELOY ALFARO (DURAN),SOLAR 3


In [ ]:
filtrada_score.sort_values(by = 'score_general', ascending = False)

,numero_establecimiento,numero_ruc,matcheo_qgram,score_qgram_02,nombre_fantasia_comercial,calles,porcentaje_match_02,score_general
3298960,21.0,9.900112e+11,DEPRATI,2.0,deprati,JOHN F. KENNEDY S/N Y AV. SUCRE AV. DE LA PR...,1.571429,3.142857
7175179,22.0,1.791288e+12,NETLIFE,2.0,netlife,AV. DE LA PRENSA S/N Y AV. MARISCAL SUCRE,1.428571,2.857143
7173214,4.0,1.791274e+12,CREPES & WAFFLES,2.0,crepes waffles,AV. DE LA PRENSA S/N Y AV. JOHN F. KENNEDY,1.428571,2.857143
7292273,2.0,1.792540e+12,LADO BUENO,2.0,lado bueno,AV. DE LA PRENSA S/N Y AV. MARISCAL SUCRE,1.428571,2.857143
7708828,6.0,1.900026e+12,FUN RIDES,2.0,fun rides,AV. DE LA PRENSA S/N Y AV. MARISCAL SUCRE,1.428571,2.857143
...,...,...,...,...,...,...,...,...
7171059,174.0,1.791256e+12,MOVISTAR,2.0,movistar,AV. LA PRENSA 6101 Y CALLE FLAVIO ALFARO,0.571429,1.142857
3360364,281.0,9.908583e+11,PHARMACYS,1.8,pharmacy s,AV. LA PRENSA N70-121,0.428571,0.771429
3416759,137.0,9.921069e+11,SWEET & COFFEE,2.0,sweet coffee,VICENTE DUQUE N77-84 Y ANTONIO CASTILLO CARCELEN,0.285714,0.571429
7124223,7.0,1.790369e+12,PRODUBANCO,2.0,produbanco,PANAMERICANA NORTE S/N Y LEONARDO MURIALDO,0.285714,0.571429


In [ ]:
percentiles = [i/10 for i in range(1,10)]

quedarnos con los que son explicitos de el recreo y luego definir un umbral